In [ ]:
# Workflow

# 1. Local: run `youtube_audio_downloader` → `./data` (raw audio)
# 2. For large datasets (e.g., 100+ hours), distribute full length audio files across `part1`, `part2`, ... under `./data` to reduce Colab timeout/disconnect risks
# 3. Upload `./data` to Google Drive (`PROJECT_DIR/data`)
# 4. Colab: run `batch_process.ipynb` segment audio files from each part → `PROJECT_DIR/segments/partN`
# 5. Colab: run `merge_segment_parts.ipynb` to merge all segment folders → `PROJECT_DIR/segments/all_files`
# 6. Colab: run `normalization_and_hf_push.ipynb` to text normalization → HF dataset creation and upload

In [ ]:
# --- merge parts ---

#CELL1 - Mount Google Drive (merge parts)
from google.colab import drive
import os
import shutil

# Mount Google Drive
print("📱 Connecting to Google Drive...")
drive.mount('/content/drive')
print("✓ Google Drive connected!\n")

In [ ]:
#CELL2 - Rename and Move Files Function
def rename_and_move_files(base_path, source_dirs, target_dir):
    """
    Rename and copy txt/wav file pairs from multiple Google Drive directories
    into a single target directory with sequential numbering.

    Args:
        base_path: Root directory path on Google Drive
        source_dirs: List of source directories (e.g. ['part1', 'part2', 'part3'])
        target_dir: Target directory name (e.g. 'all_files')
    """

    # Build full target path
    full_target_path = os.path.join(base_path, target_dir)

    # Create target directory
    if not os.path.exists(full_target_path):
        os.makedirs(full_target_path)
        print(f"✓ Created directory {target_dir}")

    # Global counter
    counter = 1
    total_files = 0

    # Process each source directory
    for source_dir in source_dirs:
        full_source_path = os.path.join(base_path, source_dir)

        if not os.path.exists(full_source_path):
            print(f"⚠ Warning: {source_dir} directory not found, skipping...")
            continue

        print(f"\n📁 Processing directory {source_dir}...")

        # List all files in directory
        files = os.listdir(full_source_path)

        # Find txt files and sort numerically
        txt_files = [f for f in files if f.endswith('.txt')]

        # Sort filenames numerically (1.txt, 2.txt, ...)
        try:
            txt_files.sort(key=lambda x: int(os.path.splitext(x)[0]))
        except:
            txt_files.sort()  # Fall back to alphabetical if numeric sort fails

        # For each txt file
        for txt_file in txt_files:
            # Base name without extension
            base_name = os.path.splitext(txt_file)[0]

            # Check for matching wav file
            wav_file = base_name + '.wav'

            txt_path = os.path.join(full_source_path, txt_file)
            wav_path = os.path.join(full_source_path, wav_file)

            # Warn and skip if wav is missing
            if not os.path.exists(wav_path):
                print(f"  ⚠ No matching wav file for {txt_file}, skipping...")
                continue

            # New filenames
            new_txt_name = f"{counter}.txt"
            new_wav_name = f"{counter}.wav"

            new_txt_path = os.path.join(full_target_path, new_txt_name)
            new_wav_path = os.path.join(full_target_path, new_wav_name)

            # Copy files
            try:
                shutil.copy2(txt_path, new_txt_path)
                shutil.copy2(wav_path, new_wav_path)

                if counter % 1000 == 0:  # Progress every 1000 file pairs
                    print(f"  ✓ Processed {counter} file pairs...")

                counter += 1
                total_files += 2

            except Exception as e:
                print(f"  ✗ Error processing {txt_file}: {e}")

        print(f"  ✓ {source_dir} completed - {counter - 1} total file pairs")

    print(f"\n{'='*50}")
    print(f"✅ Processing complete!")
    print(f"📊 Processed {counter - 1} file pairs ({total_files} files) total")
    print(f"📂 All files are in the '{target_dir}' directory")
    print(f"{'='*50}")


In [ ]:
#CELL3 - Run Merge Parts
# ==================== USAGE ====================
# Renumbers part1, part2, ... folders into a single directory.
# PROJECT_DIR must match batch_process.ipynb CELL2.

# Project root
PROJECT_DIR = "/content/drive/MyDrive/project_name"  # edit path — must match CELL2

BASE_PATH = f"{PROJECT_DIR}/segments"  # edit path — parent of part folders

# Part folders to merge (under BASE_PATH)
SOURCE_DIRS = ["part1", "part2","part3"]  # edit path — list the parts you processed

# Target folder name (created as BASE_PATH/all_files)
TARGET_DIR = "all_files"  # edit path

# Run merge
print("🚀 Starting file renaming and moving process...\n")
print(f"📍 Working directory: {BASE_PATH}\n")

rename_and_move_files(BASE_PATH, SOURCE_DIRS, TARGET_DIR)

print("\n💡 Tip: Original files are preserved. You can delete them manually if you want.")